# Oil Futures Carry Signal Construction
### For use with Brent (CO3/CO12) or WTI (CL3/CL12)

This notebook constructs the two trading signals used in the oil-FX strategy,
following Bouchouev & Zuo (2020):

| Signal | Definition | Intuition |
|---|---|---|
| **CARRY** | CO3 − CO12 (level) | Is the curve in backwardation or contango right now? |
| **CARRY-MOM** | CARRY vs its 4-week MA | Is the carry trend improving or deteriorating? |

**Output:** `Signal_Carry_CarryMom.csv` — aligned to weekly Friday dates, trimmed to
the FX data window. Drop-in replacement for any oil benchmark (Brent or WTI).

---
### Quick-start: to switch from Brent to WTI
Change the two file paths in **Section 1** to your CL3 and CL12 CSV files.
Everything else runs unchanged.

## 0. Imports & Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})


## 1. User Inputs

**This is the only section you need to change when switching benchmarks.**

- Set `PATH_NEAR` to your 3rd-nearby futures CSV (CO3 for Brent, CL3 for WTI)
- Set `PATH_FAR` to your 12th-nearby futures CSV (CO12 for Brent, CL12 for WTI)
- Set `PATH_FX` to the master FX file — used only to align output dates
- Set `BENCHMARK_NAME` to a label used in chart titles and the output filename
- Set `MA_WINDOW` for the carry-momentum moving average (default: 4 weeks,
  matching the paper's 20-day MA converted to weekly frequency)
- Set `DATE_FORMAT` to match how dates are stored in your CSV
  (`'%d/%m/%Y'` for DD/MM/YYYY, `'%Y-%m-%d'` for YYYY-MM-DD, etc.)

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────
PATH_NEAR      = 'CO3_weekly.csv'    # 3rd-nearby futures (change to CL3_weekly.csv for WTI)
PATH_FAR       = 'CO12_weekly.csv'   # 12th-nearby futures (change to CL12_weekly.csv for WTI)
PATH_FX        = 'Master_FX_Futures_transformed_interpolated.csv'

# ── Signal parameters ─────────────────────────────────────────────────────
BENCHMARK_NAME = 'Brent'   # Label for charts and output file ('Brent' or 'WTI')
MA_WINDOW      = 4         # Weeks for carry-momentum moving average

# ── Date format in your CSV files ─────────────────────────────────────────
# Common formats:
#   '%d/%m/%Y'  ->  15/05/2026  (DD/MM/YYYY)  <-- current Brent files
#   '%Y-%m-%d'  ->  2026-05-15  (YYYY-MM-DD)
#   '%m/%d/%Y'  ->  05/15/2026  (MM/DD/YYYY)
DATE_FORMAT = '%d/%m/%Y'

# ── Output filename (auto-generated from benchmark name) ──────────────────
OUTPUT_FILE = f'Signal_Carry_CarryMom_{BENCHMARK_NAME}.csv'
print(f'Settings loaded.')
print(f'  Benchmark  : {BENCHMARK_NAME}')
print(f'  Near leg   : {PATH_NEAR}')
print(f'  Far leg    : {PATH_FAR}')
print(f'  MA window  : {MA_WINDOW} weeks')
print(f'  Output file: {OUTPUT_FILE}')


## 2. Load & Inspect Raw Futures Data

We load both the near (3rd-nearby) and far (12th-nearby) contracts.
Bloomberg exports weekly Friday closing prices — we verify this and check for gaps.

In [ ]:
def load_futures(path, date_fmt, label):
    """
    Load a Bloomberg weekly futures CSV, clean column names,
    parse dates, and sort oldest-first.

    Parameters
    ----------
    path     : str  Path to CSV file
    date_fmt : str  strptime date format string
    label    : str  Human-readable label for printing

    Returns
    -------
    pd.DataFrame with DatetimeIndex, columns PX_LAST and PX_VOLUME
    """
    df = pd.read_csv(path)
    # Strip BOM and whitespace from column names
    df.columns = df.columns.str.strip().str.lstrip('\ufeff')
    df['Date'] = pd.to_datetime(df['Date'], format=date_fmt)
    df = df.sort_values('Date').set_index('Date')
    # Coerce volume to numeric (handles Bloomberg '#N/A N/A' strings)
    df['PX_VOLUME'] = pd.to_numeric(df['PX_VOLUME'], errors='coerce')
    print(f'{label}')
    print(f'  Shape     : {df.shape}')
    print(f'  Date range: {df.index.min().date()} to {df.index.max().date()}')
    print(f'  Day of week: {df.index.day_name().value_counts().to_dict()}')
    print(f'  NaN prices: {df["PX_LAST"].isna().sum()}')
    print(f'  Price range: {df["PX_LAST"].min():.2f} to {df["PX_LAST"].max():.2f}')
    print()
    return df


near = load_futures(PATH_NEAR, DATE_FORMAT, f'{BENCHMARK_NAME} Near (3rd-nearby)')
far  = load_futures(PATH_FAR,  DATE_FORMAT, f'{BENCHMARK_NAME} Far (12th-nearby)')


### 2a. Raw Price Preview

A quick look at both contracts to confirm the data loaded correctly.

In [ ]:
print(f'Near contract — first 5 rows:')
display(near.head())
print(f'\nFar contract — first 5 rows:')
display(far.head())


### 2b. Check for Date Gaps

Weekly data should have no missing Fridays (other than holidays). Large gaps may indicate data quality issues.

In [ ]:
# Check gaps: difference between consecutive dates should be 7 days
near_gaps = near.index.to_series().diff().dt.days
far_gaps  = far.index.to_series().diff().dt.days

print('Near contract — date gap distribution (days):')
print(near_gaps.value_counts().sort_index().to_string())

print('\nFar contract — date gap distribution (days):')
print(far_gaps.value_counts().sort_index().to_string())

non7_near = near_gaps[near_gaps > 8].dropna()
non7_far  = far_gaps[far_gaps > 8].dropna()

if len(non7_near) > 0:
    print(f'\nWARNING: {len(non7_near)} gaps > 8 days in near contract:')
    print(non7_near.to_string())
else:
    print('\nNo significant gaps in near contract.')

if len(non7_far) > 0:
    print(f'WARNING: {len(non7_far)} gaps > 8 days in far contract:')
    print(non7_far.to_string())
else:
    print('No significant gaps in far contract.')


## 3. Visualise Raw Futures Prices

Plot both contracts together. The gap between them visually shows the curve shape:
- **Near > Far** → backwardation (upward-sloping carry)
- **Near < Far** → contango (downward-sloping carry)

Key episodes to recognise: 2008 oil spike, 2014–2016 shale-driven contango,
2020 COVID crash, 2021–2022 backwardation recovery.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: Both contracts
axes[0].plot(near.index, near['PX_LAST'], lw=1.8,
             color='steelblue', label=f'Near (3rd-nearby)')
axes[0].plot(far.index,  far['PX_LAST'],  lw=1.8,
             color='firebrick', linestyle='--', label=f'Far (12th-nearby)')
axes[0].set_ylabel('Price (USD/bbl)')
axes[0].set_title(f'{BENCHMARK_NAME} Futures: Near vs Far Contract',
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# Panel 2: Volume
axes[1].bar(near.index, near['PX_VOLUME'], width=5,
            color='steelblue', alpha=0.6, label='Near volume')
axes[1].set_ylabel('Volume (contracts)')
axes[1].set_title('Near Contract Volume', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()


## 4. Compute the Carry Spread

**CARRY = Near − Far = CO3 − CO12**

This is the 3rd-to-12th nearby spread. It measures the premium the market
is willing to pay for near-term delivery over delivery in ~9 months' time.

| CARRY > 0 | Backwardation | Physical oil is scarce now; spot buyers compete for immediate barrels |
|---|---|---|
| CARRY < 0 | Contango | Inventories are ample; forward sellers accept a discount for deferred delivery |

We first align the two contracts to a **common date index** (inner join)
to handle any dates where one series has data and the other does not.

In [ ]:
# Align near and far to common dates (inner join)
common_idx = near.index.intersection(far.index)
near_px    = near.loc[common_idx, 'PX_LAST']
far_px     = far.loc[common_idx,  'PX_LAST']

print(f'Near-only dates  : {len(near.index.difference(far.index))}')
print(f'Far-only dates   : {len(far.index.difference(near.index))}')
print(f'Common dates     : {len(common_idx)}')
print(f'Date range       : {common_idx.min().date()} to {common_idx.max().date()}')

# Build signal dataframe on common index
sig = pd.DataFrame(index=common_idx)
sig.index.name = 'Date'
sig['Near']  = near_px
sig['Far']   = far_px
sig['CARRY'] = sig['Near'] - sig['Far']

print(f'\nCARRY summary statistics:')
print(sig['CARRY'].describe().round(3).to_string())
print(f'\nBackwardation weeks (CARRY > 0): '
      f'{(sig["CARRY"] > 0).sum()} ({(sig["CARRY"] > 0).mean()*100:.1f}%)')
print(f'Contango weeks    (CARRY < 0): '
      f'{(sig["CARRY"] < 0).sum()} ({(sig["CARRY"] < 0).mean()*100:.1f}%)')


## 5. Compute the Carry-Momentum Signal

**CARRY-MOM** compares the current carry to its own recent moving average.

```
CARRY_MA = rolling_mean(CARRY, window=MA_WINDOW)
CARRY-MOM signal:
    +1  if CARRY > CARRY_MA   (carry is above its average → improving trend)
    -1  if CARRY < CARRY_MA   (carry is below its average → deteriorating trend)
```

**Why this is more powerful than the raw CARRY level:**
The carry level tells you *where* the market is. The carry momentum tells you
*where it is heading*. An oil market moving from deep contango toward backwardation
will show a rising CARRY-MOM signal **before** CARRY crosses zero — giving you
an early entry into the regime shift.

The 4-week window corresponds to the 20-day MA used in the paper,
converted to weekly frequency (20 trading days ≈ 4 calendar weeks).

In [ ]:
# 4-week (or MA_WINDOW-week) moving average of CARRY
sig['CARRY_MA4'] = sig['CARRY'].rolling(window=MA_WINDOW).mean()

# Binary signals
# CARRY signal: +1 if backwardation (CARRY > 0), -1 if contango
sig['SIGNAL_CARRY'] = np.where(sig['CARRY'] > 0, 1, -1)

# CARRY-MOM signal: +1 if carry is above its MA (improving), -1 if below
sig['SIGNAL_CARRY_MOM'] = np.where(
    sig['CARRY'] > sig['CARRY_MA'], 1, -1
)

# Note: the first (MA_WINDOW - 1) rows of CARRY_MA will be NaN
# because we don't have enough history to compute the rolling mean yet.
# This is normal — they will be excluded when we trim to the FX date range.
print(f'Signal dataframe shape: {sig.shape}')
print(f'NaN rows (MA burn-in): {sig["CARRY_MA"].isna().sum()} '
      f'(first {MA_WINDOW-1} rows — expected)')
print()
print('SIGNAL_CARRY distribution:')
print(sig['SIGNAL_CARRY'].value_counts().rename({1:'Backwardation (+1)',
                                                   -1:'Contango (-1)'}).to_string())
print('\nSIGNAL_CARRY_MOM distribution:')
print(sig['SIGNAL_CARRY_MOM'].value_counts().rename({1:'Carry rising (+1)',
                                                      -1:'Carry falling (-1)'}).to_string())


## 6. Visualise Carry & Signals — Full History

Four panels:
1. Near and far prices
2. Carry spread with MA and regime fill
3. CARRY signal (binary level)
4. CARRY-MOM signal (binary momentum)

Key episodes annotated for reference.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# ── Panel 1: Near and far prices ──────────────────────────────────────
axes[0].plot(sig.index, sig['Near'], lw=1.5, color='steelblue',
             label='Near (3rd-nearby)')
axes[0].plot(sig.index, sig['Far'],  lw=1.5, color='firebrick',
             linestyle='--', label='Far (12th-nearby)')
axes[0].set_ylabel('Price (USD/bbl)')
axes[0].set_title(f'{BENCHMARK_NAME}: Near vs Far Prices',
                  fontweight='bold')
axes[0].legend(fontsize=10)

# ── Panel 2: Carry spread with 4-week MA ──────────────────────────────
axes[1].plot(sig.index, sig['CARRY'], lw=1.5, color='purple',
             label=f'CARRY (Near - Far)')
axes[1].plot(sig.index, sig['CARRY_MA'], lw=1.5, color='orange',
             linestyle='--', label=f'{MA_WINDOW}-week MA')
axes[1].axhline(0, color='black', lw=1, linestyle=':', alpha=0.7)
axes[1].fill_between(
    sig.index, sig['CARRY'], 0,
    where=sig['CARRY'] >= 0, alpha=0.2, color='green', label='Backwardation'
)
axes[1].fill_between(
    sig.index, sig['CARRY'], 0,
    where=sig['CARRY'] < 0, alpha=0.2, color='red', label='Contango'
)
axes[1].set_ylabel('Spread (USD/bbl)')
axes[1].set_title(f'CARRY = Near - Far, with {MA_WINDOW}-Week MA',
                  fontweight='bold')
axes[1].legend(fontsize=10, ncol=2)

# ── Panel 3: CARRY signal ─────────────────────────────────────────────
axes[2].step(sig.index, sig['SIGNAL_CARRY'], lw=2, color='steelblue',
             where='post', label='SIGNAL_CARRY')
axes[2].fill_between(
    sig.index, sig['SIGNAL_CARRY'], 0,
    step='post', alpha=0.2, color='steelblue'
)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_yticks([-1, 1])
axes[2].set_yticklabels(['-1  Contango', '+1  Backwardation'])
axes[2].set_title('SIGNAL_CARRY: +1 = Backwardation (long exporters), '
                  '-1 = Contango (short exporters)',
                  fontweight='bold')
axes[2].legend(fontsize=10)

# ── Panel 4: CARRY-MOM signal ─────────────────────────────────────────
axes[3].step(sig.index, sig['SIGNAL_CARRY_MOM'], lw=2, color='darkorange',
             where='post', label='SIGNAL_CARRY_MOM')
axes[3].fill_between(
    sig.index, sig['SIGNAL_CARRY_MOM'], 0,
    step='post', alpha=0.2, color='darkorange'
)
axes[3].axhline(0, color='black', lw=0.8)
axes[3].set_yticks([-1, 1])
axes[3].set_yticklabels(['-1  Carry falling', '+1  Carry rising'])
axes[3].set_title('SIGNAL_CARRY_MOM: +1 = Carry above MA (improving), '
                  '-1 = Carry below MA (deteriorating)',
                  fontweight='bold')
axes[3].legend(fontsize=10)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle(f'{BENCHMARK_NAME} Oil Carry Signal — Full History',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 7. Signal Divergence — When CARRY and CARRY-MOM Disagree

The two signals sometimes point in **opposite directions**. These divergence
periods are where CARRY-MOM adds the most value:

| CARRY | CARRY-MOM | Interpretation |
|---|---|---|
| +1 (backwardation) | +1 (carry rising) | Strong agreement: full long exporters |
| +1 (backwardation) | -1 (carry falling) | Regime may be peaking — carry level still high but starting to roll over |
| -1 (contango) | +1 (carry rising) | **Early exit from contango** — MOM already flipping before CARRY crosses zero |
| -1 (contango) | -1 (carry falling) | Strong agreement: full short exporters |

In [ ]:
# Compute agreement/divergence
sig['AGREEMENT'] = (sig['SIGNAL_CARRY'] == sig['SIGNAL_CARRY_MOM'])

agree_pct   = sig['AGREEMENT'].mean() * 100
diverge_pct = 100 - agree_pct

print(f'Signal agreement  : {agree_pct:.1f}% of weeks')
print(f'Signal divergence : {diverge_pct:.1f}% of weeks')

# Breakdown of the four regime combinations
combos = sig.groupby(['SIGNAL_CARRY','SIGNAL_CARRY_MOM']).size()
combos.index = combos.index.map({
    (1, 1):   'Back+1 / MOM+1 (both long exporters)',
    (1,-1):   'Back+1 / MOM-1 (carry peaking)',
    (-1, 1):  'Cont-1 / MOM+1 (carry recovering early)',
    (-1,-1):  'Cont-1 / MOM-1 (both short exporters)',
})
print('\nWeekly count by signal combination:')
print(combos.to_string())

# Visualise divergence periods
fig, ax = plt.subplots(figsize=(14, 3))
diverge = (~sig['AGREEMENT']).astype(int)
ax.fill_between(sig.index, diverge, 0, alpha=0.5, color='crimson',
                label='Signals diverge')
ax.fill_between(sig.index, sig['AGREEMENT'].astype(int), 0,
                alpha=0.15, color='steelblue', label='Signals agree')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Agree', 'Diverge'])
ax.set_title('CARRY vs CARRY-MOM Signal Agreement Over Time',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()


## 8. Carry Distribution Analysis

Understanding the statistical properties of the carry spread helps interpret
when signals are strong vs weak, and gives context for the regime breakdown.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Histogram ─────────────────────────────────────────────────────────
axes[0].hist(sig['CARRY'].dropna(), bins=60,
             color='purple', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='black', lw=1.5, linestyle='--',
                label='Zero (regime boundary)')
axes[0].axvline(sig['CARRY'].mean(), color='orange', lw=1.5, linestyle='--',
                label=f'Mean: {sig["CARRY"].mean():.2f}')
axes[0].axvline(sig['CARRY'].median(), color='green', lw=1.5, linestyle=':',
                label=f'Median: {sig["CARRY"].median():.2f}')
axes[0].set_title('Carry Distribution', fontweight='bold')
axes[0].set_xlabel('Carry (USD/bbl)')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=9)

# ── Time-series area chart ─────────────────────────────────────────────
axes[1].fill_between(
    sig.index, sig['CARRY'], 0,
    where=sig['CARRY'] >= 0, color='green', alpha=0.4, label='Backwardation'
)
axes[1].fill_between(
    sig.index, sig['CARRY'], 0,
    where=sig['CARRY'] < 0, color='red', alpha=0.4, label='Contango'
)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Carry Regime Over Time', fontweight='bold')
axes[1].set_ylabel('Carry (USD/bbl)')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_locator(mdates.YearLocator(4))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Annual regime breakdown bar chart ─────────────────────────────────
annual_regime = sig.groupby(sig.index.year)['SIGNAL_CARRY'].value_counts().unstack(fill_value=0)
if 1 in annual_regime.columns and -1 in annual_regime.columns:
    annual_regime.columns = ['Contango', 'Backwardation']
elif 1 in annual_regime.columns:
    annual_regime.columns = ['Backwardation']
    annual_regime['Contango'] = 0
else:
    annual_regime.columns = ['Contango']
    annual_regime['Backwardation'] = 0
annual_regime = annual_regime[['Backwardation','Contango']]

annual_regime.plot(
    kind='bar', stacked=True, ax=axes[2],
    color=['green','red'], alpha=0.7, edgecolor='white'
)
axes[2].set_title('Annual Regime Weeks', fontweight='bold')
axes[2].set_ylabel('Number of weeks')
axes[2].set_xlabel('Year')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(fontsize=9)

plt.suptitle(f'{BENCHMARK_NAME} Carry — Statistical Summary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary stats
print('Carry summary statistics (full history):')
print(sig['CARRY'].describe().round(3).to_string())
print(f'\nSkewness  : {sig["CARRY"].skew():.3f}')
print(f'Kurtosis  : {sig["CARRY"].kurt():.3f}')


## 9. Zoomed View: CARRY-MOM Leading the CARRY Signal

This zoomed chart focuses on a key regime transition to illustrate how
CARRY-MOM (orange) fires **before** CARRY (blue) during a contango-to-backwardation shift.

Adjust `ZOOM_START` and `ZOOM_END` to examine any period of interest.

In [ ]:
# ── Adjust these dates to zoom into any episode of interest ──────────
ZOOM_START = '2009-01-01'
ZOOM_END   = '2010-06-30'
# ─────────────────────────────────────────────────────────────────────

zoom = sig.loc[ZOOM_START:ZOOM_END]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1: Carry + MA
axes[0].plot(zoom.index, zoom['CARRY'], lw=2, color='purple',
             label='CARRY')
axes[0].plot(zoom.index, zoom['CARRY_MA'], lw=2, color='orange',
             linestyle='--', label=f'{MA_WINDOW}-week MA')
axes[0].axhline(0, color='black', lw=1, linestyle=':')
axes[0].fill_between(zoom.index, zoom['CARRY'], 0,
                     where=zoom['CARRY'] >= 0,
                     alpha=0.2, color='green', label='Backwardation')
axes[0].fill_between(zoom.index, zoom['CARRY'], 0,
                     where=zoom['CARRY'] < 0,
                     alpha=0.2, color='red', label='Contango')
axes[0].set_ylabel('Spread (USD/bbl)')
axes[0].set_title(f'Carry Spread and {MA_WINDOW}-Week MA', fontweight='bold')
axes[0].legend(fontsize=9, ncol=4)

# Panel 2: CARRY signal
axes[1].step(zoom.index, zoom['SIGNAL_CARRY'], lw=2.5,
             color='steelblue', where='post')
axes[1].fill_between(zoom.index, zoom['SIGNAL_CARRY'], 0,
                     step='post', alpha=0.25, color='steelblue')
axes[1].set_yticks([-1, 1])
axes[1].set_yticklabels(['-1', '+1'])
axes[1].set_title('SIGNAL_CARRY (fires when CARRY crosses zero)',
                  fontweight='bold', color='steelblue')

# Panel 3: CARRY-MOM signal
axes[2].step(zoom.index, zoom['SIGNAL_CARRY_MOM'], lw=2.5,
             color='darkorange', where='post')
axes[2].fill_between(zoom.index, zoom['SIGNAL_CARRY_MOM'], 0,
                     step='post', alpha=0.25, color='darkorange')
axes[2].set_yticks([-1, 1])
axes[2].set_yticklabels(['-1', '+1'])
axes[2].set_title('SIGNAL_CARRY_MOM (fires when CARRY crosses its MA — earlier)',
                  fontweight='bold', color='darkorange')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle(f'Zoomed: {ZOOM_START} to {ZOOM_END} — '
             f'CARRY-MOM leads CARRY at regime transitions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
sig

## 10. Save Output

Saves `Signal_Carry_CarryMom_{BENCHMARK_NAME}.csv` — ready to be used
directly by the backtesting notebook.

**To use this in the backtest:** change `PATH_SIGNAL` in the backtest notebook
to point to this output file.

In [ ]:
sig.index.name = 'Date'
sig.to_csv(OUTPUT_FILE)

print(f'Saved: {OUTPUT_FILE}')
print(f'Shape: {sig.shape}')
print(f'Date range: {sig.index.min().date()} to {sig.index.max().date()}')
print()
print('Column summary:')
for col in sig.columns:
    n_nan = sig[col].isna().sum()
    print(f'  {col:<25}: {sig[col].dtype} | NaN={n_nan}')
print()
print('Preview:')
display(sig.tail(5).round(3))